In [1]:
import os; os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import json, time, math, glob, numpy as np, torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass, asdict

In [2]:
DATA_DIR = os.path.dirname(glob.glob("/kaggle/input/**/train_ids_bpe.npy", recursive=True)[0])
train_ids = np.load(os.path.join(DATA_DIR, "train_ids_bpe.npy"), mmap_mode="r")
val_ids   = np.load(os.path.join(DATA_DIR, "val_ids_bpe.npy"),   mmap_mode="r")
VOCAB_SIZE = 65355
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"train {len(train_ids):,} | val {len(val_ids):,} | vocab {VOCAB_SIZE} | {device}")

train 200,000,000 | val 5,000,000 | vocab 65355 | cuda


In [3]:
def get_batch(split, B, T=1024):
    data = train_ids if split == "train" else val_ids
    starts = np.random.randint(0, len(data) - T - 1, size=B)
    x = np.stack([data[s:s+T] for s in starts]).astype(np.int64)
    y = np.stack([data[s+1:s+T+1] for s in starts]).astype(np.int64)
    return torch.from_numpy(x).to(device), torch.from_numpy(y).to(device)

In [4]:
class RMSNorm(nn.Module):
    def __init__(self,d_model,eps=1e-6):
        super().__init__()
        self.eps=eps
        self.g=nn.Parameter(torch.ones(d_model))
    def forward(self,x):
        meanSquare=x.pow(2).mean(dim=-1,keepdim=True)
        x=x*torch.rsqrt(meanSquare+self.eps)
        return x*self.g

In [5]:
def rope_tables(max_seq_len,d_head,base=10000.0):
    assert d_head%2==0
    i=torch.arange(0,d_head//2).float()
    theta=base**(-2*i/d_head)
    m=torch.arange(max_seq_len).float()
    angles=torch.outer(m,theta)
    return torch.cos(angles),torch.sin(angles)

In [6]:
def apply_rope(x,cos,sin):
    T=x.size(2)
    cos,sin=cos[:T],sin[:T]
    x1=x[...,0::2]
    x2=x[...,1::2]
    r1=x1*cos-x2*sin
    r2=x1*sin+x2*cos
    out=torch.stack([r1,r2],dim=-1)
    return out.flatten(-2)

In [7]:
class GQAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0, rope_base=10000.0):
        super().__init__()
        assert d_model%n_heads==0 and n_heads%n_kv_heads==0
        self.n_heads=n_heads
        self.n_kv_heads=n_kv_heads
        self.d_head=d_model // n_heads
        self.n_rep=n_heads // n_kv_heads
        self.scale=1/math.sqrt(self.d_head)
        self.q_proj=nn.Linear(d_model,d_model,bias=False)
        self.k_proj=nn.Linear(d_model,n_kv_heads*self.d_head,bias=False)
        self.v_proj=nn.Linear(d_model,n_kv_heads*self.d_head,bias=False)
        self.out_proj=nn.Linear(d_model,d_model,bias=False)
        self.attn_dropout=nn.Dropout(dropout)
        self.resid_dropout=nn.Dropout(dropout)
        mask=torch.tril(torch.ones(max_seq_len,max_seq_len,dtype=torch.bool))
        self.register_buffer("causal_mask",mask,persistent=False)
        cos,sin=rope_tables(max_seq_len,self.d_head, rope_base)
        self.register_buffer("cos",cos,persistent=False)
        self.register_buffer("sin",sin,persistent=False)
    def forward(self,x):
        B,T,C=x.shape
        q=self.q_proj(x).view(B,T,self.n_heads,self.d_head).transpose(1,2)
        k=self.k_proj(x).view(B,T,self.n_kv_heads,self.d_head).transpose(1,2)
        v=self.v_proj(x).view(B,T,self.n_kv_heads,self.d_head).transpose(1,2)
        q=apply_rope(q,self.cos,self.sin)
        k=apply_rope(k,self.cos,self.sin)
        k=k.repeat_interleave(self.n_rep,dim=1)
        v=v.repeat_interleave(self.n_rep,dim=1)
        scores=(q*self.scale)@k.transpose(-2,-1)
        scores=scores.masked_fill(~self.causal_mask[:T,:T],float("-inf"))
        weights=self.attn_dropout(F.softmax(scores,dim=-1))
        out=(weights@v).transpose(1,2).contiguous().view(B,T,C)
        return self.resid_dropout(self.out_proj(out))

In [8]:
class SwiGLU(nn.Module):
    def __init__(self,d_model,d_ff,dropout=0.0):
        super().__init__()
        self.w1=nn.Linear(d_model,d_ff,bias=False)
        self.w3=nn.Linear(d_model,d_ff,bias=False)
        self.w2=nn.Linear(d_ff,d_model,bias=False)
        self.dropout=nn.Dropout(dropout)
    def forward(self,x):
        gate=F.silu(self.w1(x))
        return self.dropout(self.w2(gate*self.w3(x)))
        

In [9]:
class MoEFFN(nn.Module):
    def __init__(self,d_model,d_ff,n_experts,top_k,dropout=0.0):
        super().__init__()
        self.n_experts=n_experts
        self.top_k=top_k
        self.router=nn.Linear(d_model,n_experts,bias=False)
        self.experts=nn.ModuleList([SwiGLU(d_model,d_ff,dropout) for _ in range(n_experts)])
        self.aux_loss=torch.tensor(0.0)
    def forward(self,x):
        B,T,d=x.shape
        flat=x.reshape(-1,d)
        with torch.autocast("cuda",enabled=False):
            logits=self.router(flat.float())
        probs=F.softmax(logits,dim=-1)
        top_p,top_i=probs.topk(self.top_k,dim=-1)
        top_p=top_p/top_p.sum(-1,keepdim=True)
        out=torch.zeros_like(flat)
        for e,expert in enumerate(self.experts):
            tok,slot=(top_i==e).nonzero(as_tuple=True)
            if tok.numel()==0: continue
            y=expert(flat[tok])
            out.index_add_(0,tok,(y*top_p[tok,slot,None]).to(out.dtype))
        f=torch.bincount(top_i.flatten(),minlength=self.n_experts).float()/top_i.numel()
        P=probs.mean(0)
        self.aux_loss=self.n_experts*(f*P).sum()
        return out.view(B,T,d)

In [10]:
torch.manual_seed(0)
moe = MoEFFN(64, 96, n_experts=4, top_k=2).eval()
x = torch.randn(2, 5, 64)
y = moe(x)
print("shape   :", tuple(y.shape))                                                  # (2, 5, 64)
print("params  :", sum(p.numel() for p in moe.parameters()), "| expected 73984")
print("aux     :", round(moe.aux_loss.item(), 3), "(≈1 = balanced)")
with torch.no_grad():
    for e in moe.experts[1:]:
        e.load_state_dict(moe.experts[0].state_dict())
print("identical experts == single expert:", torch.allclose(moe(x), moe.experts[0](x), atol=1e-5))   # True

shape   : (2, 5, 64)
params  : 73984 | expected 73984
aux     : 1.287 (≈1 = balanced)
identical experts == single expert: True


In [11]:
@dataclass
class HybridConfig:
    vocab_size:int
    d_model:int
    n_layers:int
    n_heads:int
    n_kv_heads:int
    d_ff:int
    max_seq_len:int=1024
    dropout:float=0.0
    rope_base:float=10000.0
    n_experts:int=4
    top_k:int=2
    d_ff_expert:int=1024

class HybridBlock(nn.Module):
    def __init__(self,cfg,layer_id):
        super().__init__()
        self.norm1=RMSNorm(cfg.d_model)
        self.attn=GQAttention(cfg.d_model,cfg.n_heads,cfg.n_kv_heads,cfg.max_seq_len,cfg.dropout,cfg.rope_base)
        self.norm2=RMSNorm(cfg.d_model)
        self.is_moe=(layer_id%2==1)
        if self.is_moe:
            self.ffn=MoEFFN(cfg.d_model,cfg.d_ff_expert,cfg.n_experts,cfg.top_k,cfg.dropout)
        else:
            self.ffn=SwiGLU(cfg.d_model,cfg.d_ff,cfg.dropout)
    def forward(self,x):
        x=x+self.attn(self.norm1(x))
        x=x+self.ffn(self.norm2(x))
        return x

class HybridLM(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.cfg=cfg
        self.tok_emb=nn.Embedding(cfg.vocab_size,cfg.d_model)
        self.blocks=nn.ModuleList([HybridBlock(cfg,i) for i in range(cfg.n_layers)])
        self.norm_f=RMSNorm(cfg.d_model)
        self.head=nn.Linear(cfg.d_model,cfg.vocab_size,bias=False)
        self.head.weight=self.tok_emb.weight
        nn.init.normal_(self.tok_emb.weight,mean=0.0,std=0.02)
    def forward(self,idx):
        x=self.tok_emb(idx)
        for block in self.blocks:
            x=block(x)
        x=self.norm_f(x)
        return self.head(x)
    @property
    def aux_loss(self):
        moe=[b.ffn.aux_loss for b in self.blocks if b.is_moe]
        return sum(moe)/len(moe)

In [12]:
torch.manual_seed(0)
small = HybridConfig(vocab_size=VOCAB_SIZE, d_model=64, n_layers=2, n_heads=8, n_kv_heads=2,
                     d_ff=172, max_seq_len=64, n_experts=4, top_k=2, d_ff_expert=86)
m = HybridLM(small).eval()
x = torch.randint(0, VOCAB_SIZE, (2, 32)); y = torch.randint(0, VOCAB_SIZE, (2, 32))
with torch.no_grad(): logits = m(x)
loss = F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1))
print("layer types :", ["MoE" if b.is_moe else "dense" for b in m.blocks])             # ['dense', 'MoE']
print(f"initial loss: {loss.item():.3f}")                                               # ≈ 11.1
x2 = x.clone(); x2[:, -1] = (x2[:, -1] + 1) % VOCAB_SIZE
with torch.no_grad(): print("causal OK   :", torch.allclose(m(x2)[:, :-1], logits[:, :-1], atol=1e-4))   # True
print("params      :", sum(p.numel() for p in m.parameters()), "| expected 4302848")
print("aux loss    :", round(m.aux_loss.item(), 3))                                     # ≈ 1.0–1.5

layer types : ['dense', 'MoE']
initial loss: 11.091
causal OK   : True
params      : 4302848 | expected 4302848
aux loss    : 1.147


In [13]:
cfg = HybridConfig(vocab_size=VOCAB_SIZE, d_model=768, n_layers=16, n_heads=12, n_kv_heads=6,
                   d_ff=2048, max_seq_len=1024, dropout=0.1, n_experts=4, top_k=2, d_ff_expert=1024)
model = HybridLM(cfg).to(device)
print(f"{sum(p.numel() for p in model.parameters()):,} parameters | expected 191,800,320")
print("MoE layers:", sum(b.is_moe for b in model.blocks), "of", len(model.blocks))     # 8 of 16

191,800,320 parameters | expected 191,800,320
MoE layers: 8 of 16


In [14]:
MICRO_B = 2
ACCUM   = 244            # 2 × 244 × 1024 = 499,712 tokens/step (same as arches 1–3)
MAX_STEPS = 400          # placeholder: set after the smoke test
PEAK_LR = 6.0e-4
MIN_LR  = 6.0e-5
WARMUP  = 40
WEIGHT_DECAY = 0.1
CLIP         = 1.0
AUX_COEF     = 0.01      # Switch Transformer's load-balancing weight
EVAL_EVERY   = 25
EVAL_BATCHES = 10
LOG_EVERY    = 5
CKPT_PATH  = "/kaggle/working/arch4_ckpt.pt"
FINAL_PATH = "/kaggle/working/arch4_final.pt"
TIME_LIMIT = 10.5 * 3600
T = cfg.max_seq_len
print(f"tokens per optimizer step: {MICRO_B * ACCUM * T:,}")

tokens per optimizer step: 499,712


In [15]:
decay_params    = [p for _, p in model.named_parameters() if p.dim() >= 2]
no_decay_params = [p for _, p in model.named_parameters() if p.dim() < 2]
optimizer = torch.optim.AdamW([
    {"params": decay_params,    "weight_decay": WEIGHT_DECAY},
    {"params": no_decay_params, "weight_decay": 0.0}], lr=PEAK_LR)

def get_lr(step):
    if step < WARMUP: return PEAK_LR * (step + 1) / WARMUP
    if step >= MAX_STEPS: return MIN_LR
    r = (step - WARMUP) / max(1, MAX_STEPS - WARMUP)
    return MIN_LR + 0.5 * (PEAK_LR - MIN_LR) * (1 + math.cos(math.pi * r))

scaler = torch.amp.GradScaler("cuda")
print(f"total {sum(p.numel() for p in decay_params) + sum(p.numel() for p in no_decay_params):,} (want 191,800,320)")

total 191,800,320 (want 191,800,320)


In [16]:
@torch.no_grad()
def estimate_loss():
    model.eval()
    out = {}
    for split in ("train", "val"):
        losses = torch.zeros(EVAL_BATCHES)
        for k in range(EVAL_BATCHES):
            x, y = get_batch(split, MICRO_B, T)
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(x)
                losses[k] = F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1)).item()
        out[split] = losses.mean().item()
    model.train()
    return out

print(estimate_loss())

{'train': 11.26033878326416, 'val': 11.247184753417969}


In [17]:
history  = {"step": [], "train": [], "val": [], "lr": []}
best_val = float("inf")
tokens_per_step = MICRO_B * ACCUM * T
t0 = t_log = time.time()
model.train()

def save(step, tag, path=CKPT_PATH):
    torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                "scaler": scaler.state_dict(), "step": step, "cfg": asdict(cfg),
                "best_val": best_val, "history": history}, path)
    print(f"  saved checkpoint ({tag}) at step {step} -> {path}")

for step in range(MAX_STEPS):
    lr = get_lr(step)
    for g in optimizer.param_groups:
        g["lr"] = lr
    for _ in range(ACCUM):
        x, y = get_batch("train", MICRO_B, T)
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(x)
            ce = F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1))
            loss = (ce + AUX_COEF * model.aux_loss) / ACCUM
        scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    gnorm = torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP)
    scaler.step(optimizer); scaler.update()
    optimizer.zero_grad(set_to_none=True)

    if not math.isfinite(ce.item()):
        print(f"  NaN/inf loss at step {step}: stopping. Last good checkpoint is safe.")
        break
    if step % LOG_EVERY == 0:
        torch.cuda.synchronize()
        dt = time.time() - t_log; t_log = time.time()
        n = LOG_EVERY if step > 0 else 1
        print(f"step {step:5d} | ce {ce.item():.3f} | aux {model.aux_loss.item():.3f} | lr {lr:.2e} | "
              f"gnorm {gnorm:.2f} | {tokens_per_step*n/dt:,.0f} tok/s | peak mem {torch.cuda.max_memory_allocated()/1e9:.1f} GB")
    if (step > 0 and step % EVAL_EVERY == 0) or step == MAX_STEPS - 1:
        l = estimate_loss()
        for k, v in (("step", step), ("train", l["train"]), ("val", l["val"]), ("lr", lr)):
            history[k].append(v)
        print(f"  EVAL step {step} | train {l['train']:.3f} | val {l['val']:.3f} | val ppl {math.exp(l['val']):.1f}")
        if l["val"] < best_val:
            best_val = l["val"]; save(step, "best")
        t_log = time.time()
    if time.time() - t0 > TIME_LIMIT:
        print(f"  time limit reached at step {step}"); break

print(f"done in {(time.time()-t0)/60:.1f} min | best val {best_val:.3f}")
save(step, "final", FINAL_PATH)
with open("/kaggle/working/arch4_history.json", "w") as f:
    json.dump(history, f)

step     0 | ce 11.242 | aux 1.158 | lr 1.50e-05 | gnorm 5.24 | 4,936 tok/s | peak mem 8.1 GB
step     5 | ce 9.707 | aux 1.256 | lr 9.00e-05 | gnorm 2.88 | 4,894 tok/s | peak mem 9.7 GB
step    10 | ce 9.184 | aux 1.101 | lr 1.65e-04 | gnorm 2.29 | 4,910 tok/s | peak mem 9.7 GB
step    15 | ce 8.115 | aux 1.153 | lr 2.40e-04 | gnorm 1.72 | 4,922 tok/s | peak mem 9.7 GB
step    20 | ce 7.397 | aux 1.064 | lr 3.15e-04 | gnorm 1.07 | 4,939 tok/s | peak mem 9.7 GB
step    25 | ce 6.545 | aux 1.043 | lr 3.90e-04 | gnorm 0.51 | 4,949 tok/s | peak mem 9.7 GB
  EVAL step 25 | train 6.933 | val 6.954 | val ppl 1046.9
  saved checkpoint (best) at step 25 -> /kaggle/working/arch4_ckpt.pt
step    30 | ce 6.707 | aux 1.121 | lr 4.65e-04 | gnorm 1.09 | 4,939 tok/s | peak mem 9.7 GB
step    35 | ce 6.657 | aux 1.361 | lr 5.40e-04 | gnorm 0.56 | 4,960 tok/s | peak mem 9.7 GB
step    40 | ce 6.658 | aux 1.289 | lr 6.00e-04 | gnorm 0.73 | 4,961 tok/s | peak mem 9.7 GB
step    45 | ce 6.020 | aux 1.248 